# ML Systems Overview

This notebook frames the rest of the `other_stuff_you_should_know` sequence. The main idea is simple: once a model is mathematically correct, a large fraction of practical ML performance comes from the *system around the model*.

We will use one small synthetic Earth-science-style task throughout these notebooks: classify a `32 x 32` gridded field by whether the strongest storm cell lands in the western or eastern half of the domain. The task is intentionally simple so the systems issues stay visible.

In this notebook we compare two end-to-end training setups:

1. A **naive** setup with a single-process dataloader and unnecessary host-side overhead.
2. A **cleaned-up** setup with parallel loading, larger batches, pinned memory when available, and a more stable training loop.


In [ ]:
import math
import platform
import random
import time
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

seed = 7
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Running on: {device}')
print(f'Python platform: {platform.platform()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('CUDA not available; the notebook still runs, but GPU-specific speedups will be illustrative only.')


In [ ]:
class StormPatchDataset(Dataset):
    # Synthetic 2D fields with optional I/O and preprocessing delays.
    def __init__(self, n_samples=4096, size=32, sleep=0.0, heavy_transform=False):
        self.n_samples = n_samples
        self.size = size
        self.sleep = sleep
        self.heavy_transform = heavy_transform
        x = np.linspace(-1.0, 1.0, size, dtype=np.float32)
        self.xx, self.yy = np.meshgrid(x, x)

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        rng = np.random.default_rng(idx)
        cx = rng.uniform(-0.65, 0.65)
        cy = rng.uniform(-0.65, 0.65)
        amplitude = rng.uniform(0.6, 1.4)
        width = rng.uniform(0.15, 0.35)
        storm = amplitude * np.exp(-((self.xx - cx) ** 2 + (self.yy - cy) ** 2) / (2 * width ** 2))
        background = 0.15 * np.sin(3 * math.pi * self.xx) + 0.1 * np.cos(2 * math.pi * self.yy)
        noise = rng.normal(0.0, 0.03, size=(self.size, self.size)).astype(np.float32)
        field = (storm + background + noise).astype(np.float32)

        if self.heavy_transform:
            field = np.tanh(1.8 * field) + 0.25 * np.roll(field, 1, axis=0) - 0.1 * np.roll(field, 2, axis=1)
            field = field.astype(np.float32)

        if self.sleep:
            time.sleep(self.sleep)

        label = np.int64(cx > 0.0)
        return torch.from_numpy(field[None, ...]), torch.tensor(label)


class SmallStormCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(16 * 8 * 8, 32),
            nn.ReLU(),
            nn.Linear(32, 2),
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
@dataclass
class EpochMetrics:
    loss: float
    accuracy: float
    epoch_seconds: float
    transfer_seconds: float
    step_seconds: float
    samples_per_second: float


def make_loader(batch_size, num_workers, pin_memory, prefetch_factor=None, persistent_workers=False):
    kwargs = dict(
        dataset=StormPatchDataset(sleep=0.0015, heavy_transform=True),
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=True,
    )
    if num_workers > 0:
        kwargs['persistent_workers'] = persistent_workers
        if prefetch_factor is not None:
            kwargs['prefetch_factor'] = prefetch_factor
        if platform.system() != 'Windows':
            kwargs['multiprocessing_context'] = 'fork'
    return DataLoader(**kwargs)


def train_one_epoch(model, loader, optimizer, device):
    criterion = nn.CrossEntropyLoss()
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    transfer_seconds = 0.0
    step_seconds = 0.0
    epoch_start = time.perf_counter()

    for xb, yb in loader:
        t0 = time.perf_counter()
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        transfer_seconds += time.perf_counter() - t0

        t1 = time.perf_counter()
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        step_seconds += time.perf_counter() - t1

        total_examples += xb.size(0)
        total_loss += loss.item() * xb.size(0)
        total_correct += (logits.argmax(dim=1) == yb).sum().item()

    epoch_seconds = time.perf_counter() - epoch_start
    return EpochMetrics(
        loss=total_loss / total_examples,
        accuracy=total_correct / total_examples,
        epoch_seconds=epoch_seconds,
        transfer_seconds=transfer_seconds,
        step_seconds=step_seconds,
        samples_per_second=total_examples / epoch_seconds,
    )


## Compare Two Setups

The naive and cleaned-up loops train the same model on the same synthetic task. Only the *system choices* differ.

- The naive configuration uses `batch_size=32`, `num_workers=0`, and no pinned memory.
- The cleaned-up configuration uses a larger batch, worker processes, persistent workers, and pinned memory when a GPU is present.

When you run this notebook, do not focus only on final accuracy. Compare **epoch time**, **samples per second**, and whether the training loop looks bottlenecked by data or by compute.


In [ ]:
configs = {
    'naive': dict(batch_size=32, num_workers=0, pin_memory=False, prefetch_factor=None, persistent_workers=False),
    'cleaned_up': dict(
        batch_size=128,
        num_workers=4 if platform.system() != 'Windows' else 2,
        pin_memory=torch.cuda.is_available(),
        prefetch_factor=2,
        persistent_workers=True,
    ),
}

rows = []
for name, cfg in configs.items():
    loader = make_loader(**cfg)
    model = SmallStormCNN().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    metrics = train_one_epoch(model, loader, optimizer, device)
    row = {'config': name, **cfg, **metrics.__dict__}
    rows.append(row)

results = pd.DataFrame(rows)
results[['config', 'batch_size', 'num_workers', 'pin_memory', 'loss', 'accuracy', 'epoch_seconds', 'transfer_seconds', 'step_seconds', 'samples_per_second']]


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))

axes[0].bar(results['config'], results['epoch_seconds'], color=['#c44e52', '#55a868'])
axes[0].set_ylabel('Epoch time (s)')
axes[0].set_title('Wall-clock time')

axes[1].bar(results['config'], results['samples_per_second'], color=['#c44e52', '#55a868'])
axes[1].set_ylabel('Samples / second')
axes[1].set_title('Throughput')

for ax in axes:
    ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()


## What To Take Away

In a mature ML workflow, you should be able to answer these questions quickly:

1. Is the model waiting on the dataloader?
2. Is device transfer time becoming visible?
3. Am I using a batch size that matches the hardware?
4. Can I reproduce the run later and explain why it was fast or slow?

The next notebooks unpack those ideas one at a time: dataloading first, then device/GPU usage, then workflow structure, then experiment tracking.
